# 사내 규정 챗봇 만들기



## RAG 실습 준비

In [ ]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os
from getpass import getpass
from dotenv import load_dotenv

/Users/yoonwogud/Desktop/Git/DeepLearning/Part3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
# os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
# os.environ["OPENAI_API_KEY"] = openai_api_key

True

In [3]:
# 실습용 가상 데이터 생성 (사내 규정 문서)
# 실제로는 PDF나 txt 파일이 있는 경우가 많지만, 실습 편의를 위해 즉석에서 파일을 만듭니다.

# 실습용 데이터 (제주도 맛집 가이드)
# jeju_content = """
# [제주도 맛집 & 여행 가이드]
# 1. 흑돼지 거리: 제주시 건입동에 위치. 저녁 7시 이후에는 웨이팅이 1시간 이상 발생할 수 있음. 
#    추천 메뉴는 흑돼지 오겹살(200g 2만원). 멜젓에 찍어 먹는 것이 특징.
   
# 2. 우도 땅콩 아이스크림: 우도 검멀레 해변 앞이 원조. 가격은 5,000원. 
#    고소한 땅콩 가루가 듬뿍 뿌려져 있음. 배 시간 때문에 오후 4시 이전에 가는 것을 추천.
   
# 3. 성산일출봉: 입장료 5,000원. 왕복 소요 시간 50분. 
#    매월 첫째 주 월요일은 휴관. 새벽 일출을 보려면 전날 근처 숙소 예약 필수.
# """

# # 질문하기
# query = "흑돼지 먹으려면 어디로 가야 해? 그리고 성산일출봉 입장료는 얼마야?"

policy_content = """
[주식회사 랭체인 인사 규정]
제1조 (근무 시간)
1. 근무 시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연 근무제를 시행하며, 코어 타임(오전 11시~오후 3시)을 준수해야 한다.

제2조 (휴가)
1. 연차 휴가는 입사 1년 후 15일이 발생한다.
2. 3년 이상 근속 시 안식월 1개월을 유급으로 제공한다.
3. 반차, 반반차 사용이 가능하며 당일 신청도 가능하다.
4. 여름 휴가는 7월~8월 중 5일을 별도로 제공한다.

제3조 (복지)
1. 도서 구입비는 월 10만원 한도 내에서 무제한 지원한다.
2. 야근 시 택시비와 식대는 법인 카드로 결제한다.
"""

# 파일을 저장합니다.
with open("company_policy.txt", "w", encoding="utf-8") as f:
    f.write(policy_content)

print("[준비 완료] 사내 규정 문서가 생성되었습니다.")

[준비 완료] 사내 규정 문서가 생성되었습니다.


## RAG 실습 시작 - LangChain 사용

### 문서 로딩 (Loading)

In [7]:
# 1. 문서 로딩
print("\n 문서를 불러옵니다...")
# from langchain_community.document_loaders import TextLoader 여기서
# pdf, csv, word, json, 웹페이지, 유튜브 자막, 위키피디아, 노션, 깃, 폴더 걍 다 됨
loader = TextLoader("company_policy.txt", encoding="utf-8") # 어떤 파일 읽을지만 설정
docs = loader.load() # 실제 파일 읽어옴
print(f"   -> 문서 길이: {len(docs[0].page_content)}자")
print(docs[0].page_content)
print(docs[0].metadata)


 문서를 불러옵니다...
   -> 문서 길이: 321자

[주식회사 랭체인 인사 규정]
제1조 (근무 시간)
1. 근무 시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연 근무제를 시행하며, 코어 타임(오전 11시~오후 3시)을 준수해야 한다.

제2조 (휴가)
1. 연차 휴가는 입사 1년 후 15일이 발생한다.
2. 3년 이상 근속 시 안식월 1개월을 유급으로 제공한다.
3. 반차, 반반차 사용이 가능하며 당일 신청도 가능하다.
4. 여름 휴가는 7월~8월 중 5일을 별도로 제공한다.

제3조 (복지)
1. 도서 구입비는 월 10만원 한도 내에서 무제한 지원한다.
2. 야근 시 택시비와 식대는 법인 카드로 결제한다.

{'source': 'company_policy.txt'}


### 문서 분할 (Splitting)

In [33]:
# 2. 문서 분할
print("\n 문서를 작은 조각(Chunk)으로 나눕니다...")
# chunk_size=100: 100자 단위로 자름
# RecursiveCharacterTextSplitter: 무식하게 자르지말고 \n기준이나 문장 단위로 잘라
# chunk_overlap=20: 문맥이 끊기지 않도록 20자는 겹치게 함
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
# LangChain Splitter: 자른 조각(Chunk)마다 **원본 파일명, 페이지 번호 같은 정보(Metadata)를 그대로 복사해서 붙여줍니다. 나중에 "출처: 사내규정 3페이지"라고 알려줄 수 있는 이유입니다.

splits = text_splitter.split_documents(docs) # 실제로 자르는 곳
print(f"   -> 총 {len(splits)}개의 조각(Chunk)으로 분할되었습니다.")

print("\n",splits[0].page_content) # 이렇게 5개로 나뉘어서 들어가있음


 문서를 작은 조각(Chunk)으로 나눕니다...
   -> 총 5개의 조각(Chunk)으로 분할되었습니다.

 [주식회사 랭체인 인사 규정]
제1조 (근무 시간)
1. 근무 시간은 오전 9시부터 오후 6시까지로 한다.


### 임베딩 & 벡터 저장소 생성 (Indexing)

In [7]:
# OpenAI의 가성비 임베딩 모델 사용
# chroma라는 VecotrDB에 저장
# 일반 db는 "단어"가 비슷한지, 벡터 디비는 의미가 비슷한지 계산해서 찾음
vectorstore = Chroma.from_documents(
    documents=splits, # 아까 잘라둔 리스트
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
)
# 토큰화 -> 임베딩을 해서 -> chroma에 저장
# 크로마에는 임베딩 벡터, 원본 텍스트, 메타데이터(출처 표기할때 사용) 저장함
print("   -> Chroma DB에 저장 완료!")


 문서를 벡터로 변환하여 저장합니다...
   -> Chroma DB에 저장 완료!


### 검색기(Retriever) 설정 & 생성 (Generation)

In [8]:
# 검색기 설정 (가장 유사한 문서 3개를 뽑아오도록 설정)
# 리트리버가 물어옴. splits중에서 가장 가까운 청크 3개를 가져옴
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# LLM 모델 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


 RAG 체인을 생성합니다...


In [ ]:
# 프롬프트 템플릿 작성
# {context}: 검색된 문서 내용이 들어갈 자리
# {question}: 사용자 질문이 들어갈 자리
template = """
당신은 회사의 인사 담당 AI입니다.
아래 [규정]을 참고하여 직원의 질문에 친절하게 답해주세요.
규정에 없는 내용은 "규정에 나와있지 않습니다"라고 답하세요.

[규정]
{context}

[질문]: {question}
"""
prompt = PromptTemplate.from_template(template)
# 문자열을 템플릿 객체로 업그레이드

In [34]:
# 체인 연결 (Retrieval -> Prompt -> LLM -> Parser): LangChain Expression Language(LCEL)
# RunnablePassthrough:질문 그대로 쓰고 싶은 경우, RunnableLambda:질문을 가공하고 싶을 경우
rag_chain = (
#retriever: 리트리버가 물어온 3개의 답변    # RunnablePassthrough: 사용자가 물어본 질문
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt # 위에서 만든 prompt에 context와 question을 넣음
    | llm   # LLM(gpt4o-mini)에 prompt를 넣음
    # 결과: AI의 답변 객체 (AIMessage(content="휴가는 15일입니다."))
    | StrOutputParser()
    # 결과: "휴가는 15일입니다." (최종 문자열)
)




### 실행 (Test)

In [30]:
# 실행
question = "여름 휴가는 언제야"

print(f"\n 질문: {question}")
print("AI 답변 생성 중...\n")

response = rag_chain.invoke(question)

print("="*30)
print(response)
print("="*30)


 질문: 여름 휴가는 언제야
AI 답변 생성 중...

여름 휴가는 7월~8월 중 5일을 별도로 제공하고 있습니다.


### 마무리 정리

In [35]:
# 실습 종료 후 임시 파일 삭제
os.remove("company_policy.txt")